# 3. Multi-horizon evaluation

The first two notebooks established the reconstruction and selected the dynamic multiscale improvement at a 96-step horizon. Here we keep that decision fixed and test whether it generalizes to the four forecast horizons used in the paper: 96, 192, 336, and 720 steps.

This is a robustness experiment, not another model-selection stage.

## 1. Setup

All training is launched from this notebook. Completed runs are saved after every model so an interrupted experiment can continue without repeating finished work.

In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ts_project.baselines import seasonal_naive_forecast
from ts_project.data import prepare_weather, build_weather_window_datasets
from ts_project.models import (
    DLinear,
    DynamicPerVariableMultiScaleDLinear,
    initialize_projections_from_dlinear,
)
from ts_project.training import seed_everything, train_forecaster

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "weather.csv"
RESULT_ROOT = PROJECT_ROOT / "results" / "dlinear" / "weather"
RUNS_PATH = RESULT_ROOT / "multihorizon_by_seed.csv"

INPUT_LENGTH = 336
HORIZONS = [96, 192, 336, 720]
SEEDS = [2021, 2022, 2023]
BATCH_SIZE = 16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Results file:", RUNS_PATH)

## 2. Experimental protocol

- Dataset, chronological split, and train-only scaling are unchanged.
- The input length remains 336 observations for every experiment.
- Only the prediction length changes.
- We compare the daily seasonal-naive baseline, reconstructed DLinear, and the already-selected V2A model.
- DLinear and V2A use the same three paired seeds, optimizer, loss, early stopping rule, and metrics.
- Seed 2021 matches the official implementation; seeds 2022 and 2023 are our robustness repetitions.
- We do not tune the architecture separately for the new horizons.

In [ ]:
weather = prepare_weather(DATA_PATH)

design = pd.DataFrame(
    {
        "horizon": HORIZONS,
        "forecast steps": HORIZONS,
        "forecast hours": [h / 6 for h in HORIZONS],
        "input steps": INPUT_LENGTH,
        "input hours": INPUT_LENGTH / 6,
    }
)
design

Weather is sampled every 10 minutes, so the longest horizon predicts five days at once from the previous 56 hours.

## 3. Evaluation and model helpers

The following functions keep the experiment compact. Test data is only evaluated after training has restored the checkpoint with the best validation MSE.

In [ ]:
def make_loaders(horizon):
    datasets = build_weather_window_datasets(
        weather,
        input_length=INPUT_LENGTH,
        prediction_length=horizon,
    )
    return {
        "train": DataLoader(datasets["train"], batch_size=BATCH_SIZE, shuffle=True, drop_last=True),
        "validation": DataLoader(datasets["validation"], batch_size=BATCH_SIZE, shuffle=True, drop_last=True),
        "test": DataLoader(datasets["test"], batch_size=BATCH_SIZE, shuffle=False),
    }


@torch.inference_mode()
def evaluate_model(model, loader):
    model.eval()
    squared_sum = absolute_sum = 0.0
    count = 0
    for inputs, targets in loader:
        predictions = model(inputs.to(DEVICE)).cpu()
        errors = predictions - targets
        squared_sum += errors.square().sum().item()
        absolute_sum += errors.abs().sum().item()
        count += errors.numel()
    mse = squared_sum / count
    return {"mse": mse, "mae": absolute_sum / count, "rmse": mse**0.5}


@torch.inference_mode()
def evaluate_seasonal_naive(loader, horizon):
    squared_sum = absolute_sum = 0.0
    count = 0
    for inputs, targets in loader:
        predictions = seasonal_naive_forecast(inputs, horizon, season_length=144)
        errors = predictions - targets
        squared_sum += errors.square().sum().item()
        absolute_sum += errors.abs().sum().item()
        count += errors.numel()
    mse = squared_sum / count
    return {"mse": mse, "mae": absolute_sum / count, "rmse": mse**0.5}

In [ ]:
def build_model(model_name, horizon, seed):
    seed_everything(seed)
    original = DLinear(
        input_length=INPUT_LENGTH,
        prediction_length=horizon,
        channels=len(weather.channel_names),
        moving_average=25,
        individual=False,
    )
    if model_name == "DLinear":
        return original

    seed_everything(seed)
    improved = DynamicPerVariableMultiScaleDLinear(
        input_length=INPUT_LENGTH,
        prediction_length=horizon,
        channels=len(weather.channel_names),
        kernel_sizes=(25, 73, 145),
        hidden_dimension=8,
    )
    initialize_projections_from_dlinear(improved, original)
    return improved


def run_trained_model(model_name, horizon, seed, loaders):
    model = build_model(model_name, horizon, seed)
    seed_everything(seed)
    started = time.perf_counter()
    training = train_forecaster(
        model,
        loaders["train"],
        loaders["validation"],
        device=DEVICE,
        learning_rate=1e-4,
        max_epochs=10,
        patience=3,
        verbose=True,
    )
    validation = evaluate_model(model, loaders["validation"])
    test = evaluate_model(model, loaders["test"])
    return {
        "horizon": horizon,
        "seed": str(seed),
        "model": model_name,
        "validation_mse": validation["mse"],
        "validation_mae": validation["mae"],
        "test_mse": test["mse"],
        "test_mae": test["mae"],
        "test_rmse": test["rmse"],
        "best_epoch": training.best_epoch,
        "parameters": sum(parameter.numel() for parameter in model.parameters()),
        "runtime_seconds": time.perf_counter() - started,
    }

## 4. Run the experiments

Running the next cell starts the full experiment. It may take roughly 30 to 90 minutes on the project GPU. Results are saved after each trained model. If execution stops, rerun the cell and completed model-seed-horizon combinations will be skipped.

In [ ]:
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
if RUNS_PATH.exists():
    records = pd.read_csv(RUNS_PATH).to_dict("records")
else:
    records = []

completed = {
    (int(row["horizon"]), str(row["model"]), str(row["seed"]))
    for row in records
}

for horizon in HORIZONS:
    print(f"\n===== Horizon {horizon} =====")
    loaders = make_loaders(horizon)

    baseline_key = (horizon, "Seasonal naive", "baseline")
    if baseline_key not in completed:
        baseline = evaluate_seasonal_naive(loaders["test"], horizon)
        records.append(
            {
                "horizon": horizon,
                "seed": "baseline",
                "model": "Seasonal naive",
                "validation_mse": np.nan,
                "validation_mae": np.nan,
                "test_mse": baseline["mse"],
                "test_mae": baseline["mae"],
                "test_rmse": baseline["rmse"],
                "best_epoch": np.nan,
                "parameters": 0,
                "runtime_seconds": 0.0,
            }
        )
        completed.add(baseline_key)

    for model_name in ("DLinear", "Dynamic multiscale V2A"):
        for seed in SEEDS:
            key = (horizon, model_name, str(seed))
            if key in completed:
                print("Already complete:", key)
                continue
            print(f"\n{model_name} | horizon {horizon} | seed {seed}")
            record = run_trained_model(model_name, horizon, seed, loaders)
            records.append(record)
            completed.add(key)
            pd.DataFrame(records).to_csv(RUNS_PATH, index=False)
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

runs = pd.DataFrame(records).sort_values(["horizon", "model", "seed"])
runs.to_csv(RUNS_PATH, index=False)
runs

## 5. Summarize the repeated-seed results

The seasonal-naive baseline has no random seed. For the two learned models, we report the mean and sample standard deviation across the three seeds.

In [ ]:
trained_runs = runs[runs["model"] != "Seasonal naive"].copy()
summary = (
    trained_runs.groupby(["horizon", "model"], as_index=False)
    .agg(
        test_mse_mean=("test_mse", "mean"),
        test_mse_std=("test_mse", "std"),
        test_mae_mean=("test_mae", "mean"),
        test_mae_std=("test_mae", "std"),
        runtime_minutes_mean=("runtime_seconds", lambda values: values.mean() / 60),
    )
)
summary

In [ ]:
baseline = trained_runs[trained_runs["model"] == "DLinear"]
improved = trained_runs[trained_runs["model"] == "Dynamic multiscale V2A"]
paired = baseline.merge(improved, on=["horizon", "seed"], suffixes=("_dlinear", "_v2a"))
paired["mse_improvement_percent"] = 100 * (
    paired["test_mse_dlinear"] - paired["test_mse_v2a"]
) / paired["test_mse_dlinear"]
paired["mae_improvement_percent"] = 100 * (
    paired["test_mae_dlinear"] - paired["test_mae_v2a"]
) / paired["test_mae_dlinear"]

improvement_summary = paired.groupby("horizon", as_index=False).agg(
    mse_improvement_mean=("mse_improvement_percent", "mean"),
    mse_improvement_std=("mse_improvement_percent", "std"),
    mae_improvement_mean=("mae_improvement_percent", "mean"),
    mae_improvement_std=("mae_improvement_percent", "std"),
    mse_wins=("mse_improvement_percent", lambda values: int((values > 0).sum())),
    mae_wins=("mae_improvement_percent", lambda values: int((values > 0).sum())),
)
improvement_summary

A positive improvement percentage means that V2A has lower error than DLinear. The paired comparison is important because both models use the same seed.

## 6. Visualize performance across horizons

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for model_name, group in summary.groupby("model"):
    axes[0].errorbar(
        group["horizon"],
        group["test_mse_mean"],
        yerr=group["test_mse_std"],
        marker="o",
        capsize=3,
        label=model_name,
    )
axes[0].set(title="Test MSE across forecast horizons", xlabel="Forecast horizon", ylabel="MSE")
axes[0].legend()

axes[1].axhline(0, color="black", linewidth=1)
axes[1].plot(
    improvement_summary["horizon"],
    improvement_summary["mse_improvement_mean"],
    marker="o",
)
axes[1].set(
    title="V2A improvement over DLinear",
    xlabel="Forecast horizon",
    ylabel="Paired MSE reduction (%)",
)

plt.tight_layout()
plt.show()

## 7. Interpretation

After running the experiment, we will use this section to answer three questions:

1. Does V2A outperform DLinear at every horizon?
2. Is the gain stable across all three paired seeds?
3. Does adaptive multiscale decomposition become more or less useful as the forecast becomes longer?

Only if these results add a clear and defensible conclusion will we promote the multi-horizon table into the final report.